In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from node2vec import Node2Vec
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Resolve the repository root whether this notebook is run from the repo root or notebooks/.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

club_edges = pd.read_csv(PROCESSED_DIR / "club_edges.csv")
club_nodes = pd.read_csv(PROCESSED_DIR / "club_nodes_with_clusters.csv")

print("Edges:", club_edges.shape)
print("Nodes:", club_nodes.shape)

display(club_edges.head())
display(club_nodes.head())


In [ ]:
G = nx.Graph()

# Add nodes
for _, row in club_nodes.iterrows():
    G.add_node(row["club"])

# Add weighted edges
for _, row in club_edges.iterrows():
    G.add_edge(
        row["source_club"],
        row["target_club"],
        weight=row["transfer_count"]
    )

print("Before removing isolated nodes:")
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

# Remove isolated nodes if any
isolated_nodes = list(nx.isolates(G))
print("Number of isolated nodes:", len(isolated_nodes))

G.remove_nodes_from(isolated_nodes)

print("After removing isolated nodes:")
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

In [ ]:
node2vec = Node2Vec(
    G,
    dimensions=64,       # 每个俱乐部生成 64 维向量
    walk_length=20,      # 每次随机游走长度
    num_walks=100,       # 每个节点随机游走次数
    workers=1,           # Windows/Jupyter 下 workers=1 最稳
    weight_key="weight",
    seed=42
)

model = node2vec.fit(
    window=10,
    min_count=1,
    batch_words=4
)

print("Node2Vec training finished.")

In [ ]:
embeddings = []

for club in G.nodes():
    vector = model.wv[str(club)]
    embeddings.append([club] + list(vector))

embedding_cols = [f"emb_{i}" for i in range(model.vector_size)]

club_embeddings = pd.DataFrame(
    embeddings,
    columns=["club"] + embedding_cols
)

print("Embedding table shape:", club_embeddings.shape)
display(club_embeddings.head())

In [ ]:
X = club_embeddings[embedding_cols].values

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

club_embeddings["embed_x"] = X_pca[:, 0]
club_embeddings["embed_y"] = X_pca[:, 1]

print("PCA explained variance ratio:", pca.explained_variance_ratio_)
display(club_embeddings.head())

In [ ]:
club_embeddings = club_embeddings.merge(
    club_nodes,
    on="club",
    how="left"
)

print("Merged embedding table shape:", club_embeddings.shape)
display(club_embeddings.head())

In [ ]:
club_embeddings.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    club_embeddings["embed_x"],
    club_embeddings["embed_y"],
    c=club_embeddings["louvain_cluster"],
    s=club_embeddings["pagerank"] * 50000,
    alpha=0.7,
    cmap="tab20"
)

plt.colorbar(scatter, label="Louvain Cluster")
plt.title("Node2Vec + PCA Club Embedding")
plt.xlabel("PCA dimension 1")
plt.ylabel("PCA dimension 2")
plt.tight_layout()
plt.show()

In [ ]:
top_label_clubs = club_embeddings.sort_values("pagerank", ascending=False).head(30)

plt.figure(figsize=(14, 10))

scatter = plt.scatter(
    club_embeddings["embed_x"],
    club_embeddings["embed_y"],
    c=club_embeddings["louvain_cluster"],
    s=club_embeddings["pagerank"] * 50000,
    alpha=0.65,
    cmap="tab20"
)

plt.colorbar(scatter, label="Louvain Cluster")

for _, row in top_label_clubs.iterrows():
    plt.text(
        row["embed_x"],
        row["embed_y"],
        row["club"],
        fontsize=8
    )

plt.title("Top Clubs in Node2Vec + PCA Embedding Space")
plt.xlabel("PCA dimension 1")
plt.ylabel("PCA dimension 2")
plt.tight_layout()
plt.show()

In [ ]:
louvain_summary = (
    club_embeddings
    .groupby("louvain_cluster")
    .agg(
        number_of_clubs=("club", "count"),
        average_pagerank=("pagerank", "mean"),
        average_degree=("degree", "mean"),
        average_weighted_degree=("weighted_degree", "mean")
    )
    .reset_index()
    .sort_values("number_of_clubs", ascending=False)
)

display(louvain_summary)

In [ ]:
for cluster_id in sorted(club_embeddings["louvain_cluster"].unique()):
    print(f"\nLouvain Cluster {cluster_id}")
    display(
        club_embeddings[
            club_embeddings["louvain_cluster"] == cluster_id
        ]
        .sort_values("pagerank", ascending=False)
        [
            [
                "club",
                "country",
                "pagerank",
                "degree",
                "weighted_degree",
                "transfers_in",
                "transfers_out"
            ]
        ]
        .head(10)
    )

In [ ]:
plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    club_embeddings["embed_x"],
    club_embeddings["embed_y"],
    c=club_embeddings["louvain_cluster"],
    s=club_embeddings["pagerank"] * 50000,
    alpha=0.7,
    cmap="tab20"
)

plt.colorbar(scatter, label="Louvain Cluster")
plt.title("Node2Vec + PCA Club Embedding Coloured by Louvain Cluster")
plt.xlabel("PCA dimension 1")
plt.ylabel("PCA dimension 2")
plt.tight_layout()
plt.show()

In [ ]:
top_label_clubs = club_embeddings.sort_values("pagerank", ascending=False).head(30)

plt.figure(figsize=(14, 10))

scatter = plt.scatter(
    club_embeddings["embed_x"],
    club_embeddings["embed_y"],
    c=club_embeddings["louvain_cluster"],
    s=club_embeddings["pagerank"] * 50000,
    alpha=0.65,
    cmap="tab20"
)

plt.colorbar(scatter, label="Louvain Cluster")

for _, row in top_label_clubs.iterrows():
    plt.text(
        row["embed_x"],
        row["embed_y"],
        row["club"],
        fontsize=8
    )

plt.title("Top Clubs in Node2Vec + PCA Embedding Space")
plt.xlabel("PCA dimension 1")
plt.ylabel("PCA dimension 2")
plt.tight_layout()
plt.show()

In [ ]:
club_embeddings.to_csv(PROCESSED_DIR / "club_embeddings.csv", index=False)
louvain_summary.to_csv(PROCESSED_DIR / "louvain_cluster_summary.csv", index=False)

print("Saved:")
print("club_embeddings.csv")
print("louvain_cluster_summary.csv")
